# Landing e preparação de ambiente
Organização do catalogo de esquemas e volumes. Alem da verificação dos arquivos .csv utilizados. 

In [0]:
#config.py possui as constates.
from config import (
    FILE_NAMES_ORIGINAL,
    CATALOG,
    LANDING_PATH,
    BRONZE_SCHEMA,
    SILVER_SCHEMA,
    GOLD_SCHEMA,
)

In [0]:
#criando o catalogo os schemas e um volume.
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {BRONZE_SCHEMA}.Landing")

In [0]:
%sql
USE CATALOG v_credit_medallion;
USE SCHEMA default;
-- Garantindo que todos os arquivos estão presentes
-- ATENÇÂO: Caso a tabela esteja vazia, certifique-se de fazer upload dos dados no volume
LIST "/Volumes/v_credit_medallion/bronze/landing/"

In [0]:
arquivos_landing = _sqldf
display(arquivos_landing)

expected_files_df = spark.createDataFrame(
    [(value,) for value in FILE_NAMES_ORIGINAL], schema="name STRING"
)
missing_df = expected_files_df.join(
    arquivos_landing.select("name"), on="name", how="left_anti"
).distinct()
missing_values = [row["name"] for row in missing_df.collect()]
if missing_values:
    raise ValueError(f"Missing files: {missing_values}")
else:
    print("[OK] Todos os arquivos esperados estão na landing zone:")
    # adicionar a leitura dos tamanhos no final